# 13.05 Microsoft / Azure — Azure OpenAI · AI Search · Cosmos DB 카탈로그

Microsoft 스택의 LangChain 통합은 두 패키지로 갈린다.

- `langchain-openai` 의 `Azure*` 클래스 — Azure OpenAI Service(채팅·임베딩) 의 정식 진입점
- `langchain-azure-ai` — Azure AI Foundry 모델 카탈로그(Mistral, Llama 등 비-OpenAI), Azure AI Search, Cosmos DB 통합

## 학습 목표

- 두 패키지(`langchain-openai` Azure 클래스 vs. `langchain-azure-ai`) 의 역할 분담을 이해한다
- Azure OpenAI 의 **deployment 이름** 개념(모델 ID 와 다름) 을 잡는다
- Azure AI Search 의 **하이브리드(시맨틱+키워드) 검색** 차별점을 식별한다

## 언제 이 공급자를 고르나

- 이미 Azure 에서 운영 중이고 **EntraID · Private Link · 데이터 거주 보장**이 필요할 때
- OpenAI 모델을 **유럽/한국 리전**에서 받고 싶을 때(Azure OpenAI 가 OpenAI 직접 API 보다 리전 다양)
- Azure AI Search 의 시맨틱 + 키워드 hybrid 를 그대로 쓰고 싶을 때
- Cosmos DB 를 LangGraph checkpointer / MongoDB 호환 vector store 로 활용하고 싶을 때

## 13.05.1 환경 설정

필요 키(어느 한쪽 이상): `AZURE_OPENAI_API_KEY` + `AZURE_OPENAI_ENDPOINT` + `AZURE_OPENAI_API_VERSION` (또는 `AZURE_AI_SEARCH_API_KEY` + `AZURE_AI_SEARCH_ENDPOINT`).

Azure OpenAI 는 **deployment 이름**(콘솔에서 직접 명명) 으로 호출한다 — `model="gpt-4.1"` 가 아니라 `azure_deployment="my-gpt-4.1-deploy"`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
assert os.environ.get("AZURE_OPENAI_API_KEY") or os.environ.get("AZURE_AI_SEARCH_API_KEY"), (
    "AZURE_OPENAI_API_KEY 또는 AZURE_AI_SEARCH_API_KEY 가 .env 에 필요합니다"
)

from langchain_openai import AzureChatOpenAI

probe = AzureChatOpenAI(
    azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    max_tokens=64,
)
print(probe.invoke("ping").content[:80])

## 13.05.2 공급자 제품군 전체 지도

| 영역 | 심볼 | 패키지 | 비고 / 링크 |
|------|------|--------|-------------|
| Chat (Azure OpenAI) | `AzureChatOpenAI` | `langchain-openai` | OpenAI 모델을 Azure 에서 |
| Chat (Azure AI Foundry) | `AzureAIChatCompletionsModel` | `langchain-azure-ai` | Mistral/Llama 등 비-OpenAI 모델 카탈로그 |
| Embeddings | `AzureOpenAIEmbeddings` | `langchain-openai` | text-embedding-3-large 등 |
| Vector store / 검색 | `AzureSearch` | `langchain-community` | Azure AI Search hybrid |
| Retriever | `AzureAISearchRetriever` | `langchain-community` | → `05_retrievers/05_vendor_managed.ipynb` |
| Cosmos DB Mongo vCore | `AzureCosmosDBVectorSearch` | `langchain-community` | MongoDB 호환 vector |
| Cosmos DB NoSQL | `AzureCosmosDBNoSqlVectorSearch` | `langchain-community` | NoSQL API + vector |
| Loader (Blob file) | `AzureBlobStorageFileLoader` | `langchain-community` | → `04_document_loaders/03_cloud_storage.ipynb` |
| Loader (Blob container) | `AzureBlobStorageContainerLoader` | `langchain-community` | container 단위 |
| Checkpointer | `CosmosDBSaver` (커뮤니티) | `langgraph` ecosystem | LangGraph 영속 상태 |

**핵심**: Azure OpenAI 채팅·임베딩은 `langchain-openai` 의 `Azure*` 클래스, AI Search · Cosmos DB · Blob Storage 는 `langchain-community` 또는 `langchain-azure-ai`.

## 13.05.3 기본 사용 — chat

`AzureChatOpenAI` 는 OpenAI 직접 API 와 거의 같은 인터페이스지만, 모델 식별이 **deployment 이름** 이라는 점이 핵심 차이.

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o-mini",   # 콘솔에서 정한 이름. 모델 ID 가 아님
    api_version="2024-10-21",
    temperature=0,
    max_tokens=256,
)
msg = llm.invoke("한국어로 Azure OpenAI 와 OpenAI 직접 API 의 차이를 한 문장으로")
print(msg.content)

## 13.05.4 공급자 특화 기능

### Azure AI Search hybrid (semantic + keyword)

Azure AI Search 는 BM25 키워드 + 벡터 + **시맨틱 reranker** 를 한 인덱스에 결합한다. RRF(Reciprocal Rank Fusion) 로 두 점수를 합치고, 시맨틱 모드를 켜면 Microsoft 가 학습한 reranker 가 상위 50건을 다시 정렬.

```python
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import AzureOpenAIEmbeddings

store = AzureSearch(
    azure_search_endpoint=os.environ["AZURE_AI_SEARCH_ENDPOINT"],
    azure_search_key=os.environ["AZURE_AI_SEARCH_API_KEY"],
    index_name="docs",
    embedding_function=AzureOpenAIEmbeddings(
        azure_deployment="text-embedding-3-large",
        api_version="2024-10-21",
    ).embed_query,
)
results = store.hybrid_search(query="보안 정책", k=5)
```

### Azure OpenAI vs OpenAI 가격 차이

| 항목 | OpenAI 직접 API | Azure OpenAI |
|------|------------------|--------------|
| 단가 | 표준가 | **거의 동일** (모델별 0~5% 차이) |
| 리전 | 미국 중심 | 글로벌 — 한국 중부 / 유럽 / 아시아 가용 |
| 데이터 보존 | OpenAI 정책 | **Azure 계정 내** — 모델 학습 미사용 |
| EntraID/Private Link | 미지원 | 지원 |
| Provisioned Throughput Units (PTU) | 없음 | **확보 용량 단위 과금** 옵션 — 안정 지연 |
| 콘텐츠 필터 | OpenAI Moderation | Azure Content Safety 자동 적용(끄기 가능) |

Azure 의 PTU 는 트래픽이 안정적인 프로덕션에서 비용 예측을 쉽게 만든다.

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings

embed = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-3-large",
    api_version="2024-10-21",
)
vec = embed.embed_query("langchain 이란")
print("dim:", len(vec))

## 13.05.5 가격·한도·리전

| 항목 | 값 (2025 공개치) |
|------|------------------|
| Azure OpenAI 리전 | East US, West US, Sweden Central, Japan East, **Korea Central**(일부 모델), UAE North 등 — 모델별 가용 리전 다름 |
| 모델 배포 | 콘솔/CLI 로 deployment 생성 → API 호출 시 deployment 이름 사용 |
| 가격 (예) | gpt-4o-mini 입력 $0.150/1M, 출력 $0.600/1M (OpenAI 직접 API 와 동률) |
| PTU | Provisioned Throughput Unit — 시간당 고정 과금, 지연 SLA |
| Rate limit | TPM(토큰/분) + RPM(요청/분), deployment 단위 — 콘솔에서 조정 |
| Azure AI Search | Basic/Standard/Storage Optimized SKU. 시맨틱 모드는 별도 과금($1/1k 쿼리) |
| Cosmos DB | RU/s + 스토리지. vector 인덱스는 Mongo vCore 또는 NoSQL API 모두 지원 |
| Content filter | 기본 ON. 콘솔에서 카테고리별 임계값 조절 |

최신 단가: https://azure.microsoft.com/ko-kr/pricing/details/cognitive-services/openai-service/

## 핵심 정리

- Microsoft 스택은 **두 패키지** — `langchain-openai`(Azure OpenAI 채팅·임베딩) + `langchain-azure-ai`/`langchain-community`(검색·DB·로더)
- Azure OpenAI 의 식별 단위는 **deployment 이름** — 모델 ID 와 별개
- 차별점은 **리전 다양성 + EntraID + PTU + 시맨틱 hybrid 검색**
- 한국 리전(Korea Central) 가용 모델이 점차 확대 중

## 다음

- chat 깊은 구현: `08_integration/01_chat_models/01_openai.ipynb` 의 Azure 섹션
- 임베딩 깊은 구현: `08_integration/02_embeddings/01_openai.ipynb` (Azure 변형 포함)
- AI Search retriever: `08_integration/05_retrievers/05_vendor_managed.ipynb`
- 다음 공급자 카탈로그: `06_groq.ipynb`

## 참고

- 공식 통합 페이지: https://docs.langchain.com/oss/python/integrations/providers/microsoft
- Azure OpenAI 문서: https://learn.microsoft.com/azure/ai-services/openai/
- Azure AI Search hybrid: https://learn.microsoft.com/azure/search/hybrid-search-overview